# Project: Fairness Audit ⚖️
Audit a classifier like a regulator would. On the classic **Adult** census data (predict income > $50k), measure whether the model treats men and women differently by three fairness definitions, then fix *one* of them with group-specific thresholds and see what it costs. It connects to your IT-ret notes too: GDPR art. 22 on automated decisions.

Topic: [[Explainability and Fairness]] · guide note: [[Project - Fairness Audit]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

adult = fetch_openml("adult", version=2, as_frame=True, data_home="data")
X = adult.data.copy(); y = (adult.target == ">50K").astype(int).to_numpy()
sex = X.pop("sex").astype(str).to_numpy()            # the protected attribute is NOT a model input…
for c in X.select_dtypes("category"): X[c] = X[c].cat.codes
Xtr, Xte, ytr, yte, str_, ste = train_test_split(X, y, sex, test_size=0.3, stratify=y, random_state=0)
clf = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
scores = clf.predict_proba(Xte)[:, 1]; pred = (scores > 0.5).astype(int)
print(f"accuracy {np.mean(pred == yte):.3f}; base rate >50K: men {yte[ste == 'Male'].mean():.1%}, women {yte[ste == 'Female'].mean():.1%}")

## 1. Group rates
For each group return `selection` = P(ŷ=1), `tpr` = P(ŷ=1 | y=1), `fpr` = P(ŷ=1 | y=0). Three definitions follow:
- **Demographic parity:** equal selection rates.
- **Equal opportunity:** equal TPR (qualified people are found equally often).
- **Equalised odds:** equal TPR **and** FPR.

In [ ]:
def group_rates(y_true, y_pred, group):
    # TODO 1: dict group → {"selection", "tpr", "fpr"}
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_rates():
    r = group_rates(np.array([1, 1, 0, 0, 1, 0]), np.array([1, 0, 1, 0, 1, 1]), np.array(["a", "a", "a", "a", "b", "b"]))
    return r["a"] == dict(selection=0.5, tpr=0.5, fpr=0.5) and r["b"] == dict(selection=1.0, tpr=1.0, fpr=1.0)
check("group rates", _t_rates, "Toy group a: 2/4 selected, TPR 1/2, FPR 1/2; group b: everything 1.0.")
if ready("group rates"):
    rates = pd.DataFrame(group_rates(yte, pred, ste)).T; print(rates.round(3))
    print(f"gaps (men − women): selection {rates.selection['Male'] - rates.selection['Female']:+.3f}, TPR {rates.tpr['Male'] - rates.tpr['Female']:+.3f}, FPR {rates.fpr['Male'] - rates.fpr['Female']:+.3f}")
    print("→ Sex wasn't an input, yet the rates differ: other features (relationship, occupation, hours) act as proxies.")

## 2. Fix equal opportunity with group thresholds
Choose a separate threshold per group so that **both** groups reach the same target TPR: the threshold is the `(1 − target_tpr)` quantile of the scores of that group's positives. Return `{group: threshold}`.

In [ ]:
def equal_opportunity_thresholds(scores, y_true, group, target_tpr=0.75):
    # TODO 2: per group, the (1 - target_tpr) quantile of scores among true positives
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_thr():
    t = equal_opportunity_thresholds(np.array([0.1, 0.2, 0.3, 0.4, 0.9]), np.array([1, 1, 1, 1, 0]), np.array(["a"] * 5), target_tpr=0.75)
    return abs(t["a"] - np.quantile([0.1, 0.2, 0.3, 0.4], 0.25)) < 1e-12
check("thresholds", _t_thr, "Threshold = 25 % quantile of the positives' scores when the target TPR is 75 %.")
if ready("group rates", "thresholds"):
    thr = equal_opportunity_thresholds(scores, yte, ste)
    fair = (scores > np.vectorize(thr.get)(ste)).astype(int)
    r2 = pd.DataFrame(group_rates(yte, fair, ste)).T
    print("thresholds:", {k: round(v, 3) for k, v in thr.items()}); print(r2.round(3))
    print(f"accuracy {np.mean(pred == yte):.3f} → {np.mean(fair == yte):.3f}")
    check("TPR gap closed", lambda: abs(r2.tpr["Male"] - r2.tpr["Female"]) < 0.03, "After group thresholds the TPR gap should be < 3 points.")
    print("→ Note what happened to selection rates and FPR: you can't satisfy all definitions at once when base rates differ (Kleinberg et al., Chouldechova).")

<details><summary>▶ Solution</summary>

```python
def group_rates(y_true, y_pred, group):
    out = {}
    for g in np.unique(group):
        m = group == g
        out[g] = dict(selection=float(y_pred[m].mean()),
                      tpr=float(y_pred[m & (y_true == 1)].mean()),
                      fpr=float(y_pred[m & (y_true == 0)].mean()))
    return out
```

```python
def equal_opportunity_thresholds(scores, y_true, group, target_tpr=0.75):
    thr = {}
    for g in np.unique(group):
        pos = np.sort(scores[(group == g) & (y_true == 1)])
        thr[g] = float(np.quantile(pos, 1 - target_tpr))
    return thr
```
</details>

## Stretch goals
- Calibration within groups: is a score of 0.7 equally reliable for men and women? Plot reliability curves per group.
- Remove the proxies (relationship, marital-status) and re-measure. Does it help, and what does it cost?
- Write the audit as a one-page DPIA-style memo in Danish, using [[Lektion 4 - Sikkerhed, konsekvensanalyse, brud, DPO og tilsyn]] as the template.